# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 먼저 다음 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [2]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-study\.venv\Scripts\python.exe
프로젝트 루트: G:\내 드라이브\dev_GD\llm-data-analysis-study
전처리 데이터 폴더: G:\내 드라이브\dev_GD\llm-data-analysis-study\data\processed
보고서 폴더: G:\내 드라이브\dev_GD\llm-data-analysis-study\reports


## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [3]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (764, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [4]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,184
1,cancelled,64
2,refunded,52


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


### 데이터 품질 검증 — 날짜 오류 및 처리 근거

**오류 확인**

공식 실습 데이터에서 고객 가입일(`signup_date`)이 주문일(`order_date`)보다 늦은 주문 45건을 확인하였다. 분류 대상 248건의 약 18.15%에 해당한다.

공식 `data/raw`와 별도로 재생성한 샘플 데이터에서 동일한 오류가 재현되었다. 데이터 생성 과정에서 가입일과 주문일의 선후관계가 보장되지 않는 것이 원인으로 판단된다.

**처리 방법**

- 원본 CSV와 주문 248건은 변경하지 않았다.
- 날짜 역전 45건은 `negative_days_since_signup` 항목에 기록하였다.
- 유효하지 않은 가입 후 경과일은 `NaN`으로 표시하여 데이터 품질 검증에 활용하였다.
- 파생 변수 `days_since_signup`은 모델 입력 Feature에서 제외하였다.
- 정상적인 주문일에서 생성한 `order_month`, `order_dayofweek`는 유지하였다.

**나의 해석과 판단**

가입 후 경과일은 고객의 구매 이력을 설명하는 데 활용할 수 있으나 현재 데이터에서는 약 18%의 값이 날짜 정합성을 충족하지 못한다. 임의의 중앙값을 부여하기보다 해당 변수를 제외하여 불확실한 정보가 모델 학습에 반영되지 않도록 하였다.

이에 따라 숫자형 Feature 6개와 범주형 Feature 3개를 사용하며 Target 정의와 Train/Validation/Test 분할 및 모델·Threshold 선택 기준은 변경하지 않는다.

In [5]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)

print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)


모델링 데이터: (248, 16)
숫자형 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek']
범주형 feature: ['gender', 'city', 'payment_method']


,is_cancelled,class_label,count,ratio
0,0,completed,184,0.7419
1,1,cancelled,64,0.2581


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,248,248,True,0,PASS
1,customer_id → customers,248,248,True,0,PASS


,check_item,count
0,binary_target_rows,248
1,completed_rows,184
2,cancelled_rows,64
3,excluded_status_rows,52
4,line_total_mismatch,0
5,negative_days_since_signup,45
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,184
9,status_scope:cancelled,64


In [6]:
# Chapter 10 날짜 정합성 보정 Evidence (모델 학습 전 확인)
negative_date_count = int(
    data_quality_checks.loc[
        data_quality_checks["check_item"].eq("negative_days_since_signup"),
        "count",
    ].iloc[0]
)
imputed_input_count = int(model_data["days_since_signup"].isna().sum())
remaining_negative_count = int(model_data["days_since_signup"].lt(0).sum())

print("원본 가입일·주문일 역전 건수:", negative_date_count)
print("결측으로 표시한 days_since_signup 건수:", imputed_input_count)
print("남아 있는 음수 경과일 건수:", remaining_negative_count)

assert negative_date_count == imputed_input_count
assert remaining_negative_count == 0


원본 가입일·주문일 역전 건수: 45
결측으로 표시한 days_since_signup 건수: 45
남아 있는 음수 경과일 건수: 0


## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [7]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')


사용 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'gender', 'city', 'payment_method']
데이터 누수 컬럼 없음


## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [8]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


X_train: (148, 9)
X_valid: (50, 9)
X_test: (50, 9)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,110,0.7432
1,train,1,cancelled,38,0.2568
2,validation,0,completed,37,0.7400
3,validation,1,cancelled,13,0.2600
4,test,0,completed,37,0.7400
5,test,1,cancelled,13,0.2600


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [9]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.72,0.466667,0.538462,0.500000
1,Random Forest,validation,0.68,0.363636,0.307692,0.333333
2,Dummy Most Frequent,validation,0.74,0.000000,0.000000,0.000000


## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [10]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))


선택 모델: Logistic Regression
선택 임계값: 0.45


,threshold,accuracy,precision,recall,f1
5,0.45,0.70,0.450000,0.692308,0.545455
8,0.60,0.78,0.600000,0.461538,0.521739
4,0.40,0.62,0.384615,0.769231,0.512821
6,0.50,0.72,0.466667,0.538462,0.500000
7,0.55,0.76,0.545455,0.461538,0.500000
3,0.35,0.52,0.333333,0.846154,0.478261
2,0.30,0.44,0.307692,0.923077,0.461538
1,0.25,0.34,0.282609,1.000000,0.440678
0,0.20,0.30,0.270833,1.000000,0.426230
9,0.65,0.74,0.500000,0.307692,0.380952


## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [11]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.45,frozen_before_test,0.6,0.315789,0.461538,0.375


,pred_completed,pred_cancelled
actual_completed,24,13
actual_cancelled,7,6


,label,precision,recall,f1-score,support
0,completed,0.774194,0.648649,0.705882,37.0
1,cancelled,0.315789,0.461538,0.375000,13.0
2,accuracy,0.600000,0.600000,0.600000,0.6
3,macro avg,0.544992,0.555094,0.540441,50.0
4,weighted avg,0.655008,0.600000,0.619853,50.0


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [12]:
from src.classification import create_prediction_result

prediction_result = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)

prediction_result.head()


,record_id,source_index,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,28,0,1,0.683535,Logistic Regression,0.45
1,test_0002,146,0,0,0.409431,Logistic Regression,0.45
2,test_0003,211,0,1,0.512531,Logistic Regression,0.45
3,test_0004,125,0,1,0.602233,Logistic Regression,0.45
4,test_0005,77,0,1,0.746377,Logistic Regression,0.45


## 11. LLM 코드 검토 체크리스트


In [13]:
from src.classification import build_classification_checklist

classification_checklist = build_classification_checklist()
classification_checklist


,check_item,status
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,□
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,□
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,□
3,line_total = quantity × unit_price를 검증했는가?,□
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,□
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",□
6,"train, validation, test를 분리했는가?",□
7,모델과 threshold는 validation에서 선택했는가?,□
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,□
9,Dummy 기준 모델과 비교했는가?,□


## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


In [14]:
from src.classification import run_classification_analysis

classification_result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')


선택 모델: Logistic Regression
선택 임계값: 0.45


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.45,frozen_before_test,0.6,0.315789,0.461538,0.375


model_data_internal G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_classification_model_data_internal.csv OK
target_distribution G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_target_distribution.csv OK
feature_audit G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_feature_audit.csv OK
merge_checks G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_merge_checks.csv OK
data_quality_checks G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_data_quality_checks.csv OK
split_summary G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_split_summary.csv OK
validation_comparison G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_validation_model_comparison.csv OK
threshold_metrics G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_validation_threshold_metrics.csv OK
test_metrics G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_test_metrics.csv OK
predictions_internal G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch10_classification_predictions_internal.

## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.
```


## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.


# Chapter 10 답안 양식. 분류 분석으로 주문 취소 여부 예측하기

> 이 내용을 `chapter10.ipynb`의 Markdown 셀로 작성합니다. 실제 실행 결과를 근거로 작성하고, 예측 패턴을 원인처럼 단정하지 않습니다.

## 제출 정보
- 이름: 김희철
- GitHub ID: KHeeCheol
- 작성일: 2026-11-02
- 최종 Notebook URL: https://github.com/KHeeCheol/llm-data-analysis-study/blob/main/notebooks/chapter10.ipynb

## 1. 분류 문제 정의
- 타깃: `is_cancelled` — 주문의 취소 여부를 나타내는 이진 분류 Target
- 0의 의미: `completed` (완료 주문)
- 1의 의미: `cancelled` (취소 주문)
- 제외한 상태: `refunded` 및 기타 상태
- 예측 단위: 주문 1건 (`order_id` 기준)
- 예측 시점: 주문 생성 과정에서 주문 상품·수량·금액 및 결제수단이 확정된 이후 최종 취소 여부가 결정되기 이전 시점으로 가정한다.
- 예측 시점에 사용할 수 있는 정보: 고객의 기본 특성(`age`, `gender`, `city`), 가입 이후 경과일, 주문 날짜 정보, 결제수단 및 확정된 주문 구성·금액 정보
- 예측 이후에만 알 수 있어 제외한 정보: 최종 `order_status`, `is_cancelled`, 취소 사유 및 취소 시각 등 사후 정보. 또한 `order_id`, `customer_id`, `product_id`와 같은 식별자는 예측 Feature에서 제외한다.

### 나의 해석과 판단
이번 분석은 주문이 완료 또는 취소 중 어느 상태에 해당하는지 예측하는 이진 분류 문제로 정의하였다. `refunded`는 단순한 비취소 주문과 의미가 다르므로 0으로 일괄 변환하지 않고 분석 대상에서 제외하였다.  
또한 주문 구성과 결제수단은 확정되었지만 취소 결과는 아직 알려지지 않은 시점을 가정하여 Feature의 사용 가능 범위를 설정하였다. 주문 상세 집계값 역시 해당 시점에 확보된다는 가정이 성립할 때만 사용하며 실제 서비스에서는 정보 생성 시점에 대한 추가 확인이 필요하다.  
현재 취소 주문 비율은 25.81%로 클래스 간 차이가 존재하므로 이후 모델 평가는 Accuracy뿐 아니라 Precision, Recall, F1 및 Dummy Baseline과의 비교를 중심으로 수행한다.
---

## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature

`age`, `item_count`, `total_quantity`, `order_amount`, `order_month`, `order_dayofweek`

### 사용한 범주형 Feature

`gender`, `city`, `payment_method`

### 금지 Feature

`order_status`, `is_cancelled`, `order_id`, `customer_id`, `product_id`, 취소 사유 및 취소 이후 생성된 정보

### 데이터 품질에 따른 추가 제외 Feature

`days_since_signup`은 공식 가상 데이터에서 가입일과 주문일의 선후관계가 역전된 45건이 확인되어 모델 입력에서 제외하였다. 다만 정상 주문 날짜에서 생성한 `order_month`, `order_dayofweek`는 유지하였다.

### 나의 해석과 판단

데이터셋에 컬럼이 존재한다는 사실만으로 예측 시점에 사용할 수 있는 Feature라고 판단할 수는 없다. Target이나 사후 정보는 Data Leakage를 발생시킬 수 있으며 날짜 정합성이 확보되지 않은 파생 변수는 예측 결과의 신뢰성을 저하시킬 가능성이 있다.  
따라서 예측 시점에 이용 가능한 정보와 데이터 품질을 함께 고려하여 Numeric Feature 6개, Categorical Feature 3개를 선정하였다.
---

## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인


`line_total = quantity × unit_price`

- 계산 불일치 건수: 0건
- 주문 특징 `order_id` 중복 건수: 0건 (중복 검증 통과)

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
|---|---|---:|---:|---:|---|
| 주문 ↔ 주문 특징 | one_to_one | 248 | 248 | 0 | PASS |
| 주문 ↔ 고객 | many_to_one | 248 | 248 | 0 | PASS |

### 결과 관찰
완료·취소 주문 248건을 대상으로 주문 상세 특징과 고객 정보를 병합하였다. 두 병합 모두 행 수가 유지되었으며 미매칭은 발생하지 않았다. 주문 상세 금액의 계산 불일치도 0건으로 확인되었다.
다만 가입일이 주문일보다 늦은 주문 45건이 별도로 발견되어 해당 파생 Feature를 제외하였다.

### 나의 해석과 판단
병합 과정에서 미매칭을 원인 확인 없이 0으로 대체하면 실제 정보 누락이 특정한 수치로 해석되어 모델의 학습 결과를 왜곡할 수 있다. 따라서 병합 관계, 행 수 및 미매칭 여부를 검증한 후 분석을 진행하는 것이 적절하다.
이번 분석에서는 주문 수와 병합 관계를 유지하면서 날짜 정합성이 확보되지 않은 Feature만 제외하여 원본 주문의 손실을 방지하였다.
---

## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
|---:|---|---:|---:|
| 0 | completed | 184 | 74.19% |
| 1 | cancelled | 64 | 25.81% |

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰
분류 대상 주문은 총 248건이며 완료 주문이 184건(74.19%), 취소 주문이 64건(25.81%)으로 확인되었다. 완료 주문이 취소 주문보다 약 2.88배 많아 클래스 간 불균형이 존재한다.

### 나의 해석과 판단
모든 주문을 다수 클래스인 완료 주문으로 예측하더라도 Accuracy는 약 74.19%가 될 수 있다. 그러나 이러한 모델은 실제 취소 주문을 탐지하지 못하므로 Accuracy만으로 분류 성능을 평가하는 것은 부적절하다.
따라서 Dummy Most Frequent를 Baseline으로 설정하고 이후 Logistic Regression과 Random Forest가 Precision, Recall 및 F1 측면에서 실질적인 개선을 제공하는지 확인할 필요가 있다.
---

## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Dummy Most Frequent | 0.7400 | 0.0000 | 0.0000 | 0.0000 |
| Logistic Regression | 0.7200 | 0.4667 | 0.5385 | 0.5000 |
| Random Forest | 0.6800 | 0.3636 | 0.3077 | 0.3333 |

![Baseline 비교](images/step03_baseline.png)

- Validation에서 선택한 모델:Logistic Regression

### 결과 관찰
Dummy Most Frequent는 Accuracy 74%로 가장 높았으나 취소 주문에 대한 Recall과 F1은 모두 0이었다. Logistic Regression은 Accuracy가 72%로 다소 낮았지만, Recall 53.85%, F1 0.5000을 기록하여 비베이스라인 후보 중 가장 높은 취소 탐지 성능을 보였다. Random Forest는 F1 0.3333으로 Logistic Regression보다 낮았다.
### 나의 해석과 판단
이번 분석의 목적은 단순히 전체 주문을 정확하게 분류하는 것이 아니라 취소 가능성이 있는 주문을 식별하는 데 있다. 따라서 다수 클래스를 예측하여 높은 Accuracy를 얻는 Dummy 모델보다 실제 취소 주문을 탐지하는 Logistic Regression이 분석 목적에 더 적합하다고 판단하였다.
Logistic Regression은 Validation에서 F1을 우선하고 Recall과 Precision을 함께 고려하는 공식 선택 기준에 따라 선정하였다.
### 한계와 추가 확인 사항
Validation 데이터가 50건이며 실제 취소 주문은 13건에 불과하므로 성능 추정에 불확실성이 존재한다. 또한 Logistic Regression도 취소 주문 6건을 놓쳤고 완료 주문 8건을 취소 위험으로 예측하였다. 따라서 Threshold에 따른 오류 유형의 변화를 분석한 후 최종 사용 가능성을 판단할 필요가 있다.
---

## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
|---|---|---:|---:|
| Train | 모델 학습 | 110 | 38 |
| Validation | 모델·Threshold 선택 | 37 | 13 |
| Test | 선택 이후 최종 평가 | 37 | 13 |

### 나의 해석과 판단
전체 248건을 Train 148건, Validation 50건, Test 50건으로 분리하였다. Stratified Random Split을 적용한 결과 각 데이터셋의 취소 주문 비율이 약 26%로 유지되었으며 세 데이터셋 모두 완료·취소 클래스가 포함되었다.
Train은 모델과 전처리 파라미터 학습에 사용하고 Validation은 후보 모델 및 Threshold 선택에 사용한다. Final Test는 두 선택이 확정된 이후 일반화 성능을 평가하는 용도로만 사용한다.
Test 결과를 확인한 뒤 모델이나 Threshold를 다시 선택하면 평가 데이터에 맞춰 의사결정이 이루어져 성능이 낙관적으로 평가될 수 있다. 따라서 모델 선택과 최종 평가는 명확하게 분리해야 한다.

### 교육용 random split의 한계
이번 실습은 클래스 비율을 유지하기 위해 무작위 분할을 사용했지만 실제 미래 주문 예측에서는 과거 데이터로 학습하고 이후 기간의 주문으로 평가하는 시간순 검증이 필요하다.
또한 동일 고객의 주문이 여러 데이터셋에 포함될 가능성이 있으므로 실제 운영 전에는 고객 단위의 데이터 중복 영향과 기간별 성능 변화도 추가 확인할 필요가 있다.
---

## 7. Validation 성능과 우선 지표

- Accuracy: 0.7200
- Precision: 0.4667
- Recall: 0.5385
- F1: 0.5000
위 성능은 모델 후보 비교 단계에서 기본 Threshold 0.50을 적용한 Validation 결과이다. 

### 내가 가장 중요하게 본 지표
F1을 우선 지표로 설정하고 취소 주문 탐지 관점에서 Recall을 함께 고려하였다.

### 그 이유

> 실제 FP/FN 비용 정보가 없다면 가정임을 명시하세요.
이번 분석은 주문 취소 여부를 예측하는 문제이므로 Accuracy만으로는 모델의 유효성을 판단하기 어렵다. 특히 Dummy 모델은 Accuracy 74%에도 불구하고 취소 주문을 전혀 탐지하지 못하였다.
Logistic Regression의 Recall은 53.85%로 실제 취소 주문 13건 중 7건을 탐지하였으나 Precision은 46.67%로 오탐지 또한 존재하였다. 따라서 Precision과 Recall을 함께 반영하는 F1을 우선 기준으로 사용하는 것이 적절하다고 판단하였다.
다만 실제 업무에서는 취소 위험을 놓치는 FN과 정상 주문에 불필요한 조치를 취하는 FP의 비용이 다를 수 있다. 현재 데이터에는 구체적인 업무 비용이 없으므로 F1 우선 판단은 교육용 분석 기준이며 실제 적용 시에는 오류 비용에 따른 재검토가 필요하다.
---

## 8. Threshold 비교

| Threshold | Precision | Recall | F1 | 예상 특징 |
|---:|---:|---:|---:|---|
| 0.40 | 0.3846 | 0.7692 | 0.5128 | 취소 탐지 증가, FP 부담 증가 |
| 0.45 | 0.4500 | 0.6923 | 0.5455 | F1 최고, 탐지율과 오탐지의 균형 |
| 0.50 | 0.4667 | 0.5385 | 0.5000 | 기본 분류 기준 |
| 0.60 | 0.6000 | 0.4615 | 0.5217 | Precision 상승, Recall 감소 |


![Threshold 비교](images/step05_threshold.png)

### 내가 선택한 Threshold
0.45

### 선택 이유
Validation에서 비교한 Threshold 중 0.45가 F1 0.5455로 가장 높은 성능을 보였다. 기본값 0.50과 비교하면 Recall이 0.5385에서 0.6923으로 증가하여 실제 취소 주문 13건 중 9건을 탐지하였다.
이에 따라 FN은 6건에서 4건으로 감소했으나 FP는 8건에서 11건으로 증가하였다. 실제 FP/FN의 업무 비용이 제공되지 않았으므로 이번 실습에서는 공식 선택 기준인 Validation F1을 우선하여 0.45를 선택하였다.
### 나의 해석과 판단
Threshold는 Final Test가 아닌 Validation 데이터에서 결정해야 한다. Validation은 모델과 Threshold를 선택하는 데 사용하는 데이터인 반면 Final Test는 모든 선택이 확정된 이후 모델의 일반화 성능을 독립적으로 평가하기 위한 데이터이기 때문이다.
만약 Final Test에서 여러 Threshold를 비교하고 가장 좋은 결과를 선택한다면 Test 데이터의 특성에 맞춰 평가 기준이 최적화되어 성능이 낙관적으로 추정될 수 있다. 이 경우 새로운 데이터에서도 동일한 성능이 재현될 것이라고 판단하기 어려워진다.
따라서 이번 분석에서는 Validation 결과만으로 Logistic Regression과 Threshold 0.45를 확정하였으며 Final Test 결과를 확인한 후에도 동일한 Test 데이터를 기준으로 선택을 변경하지 않는다.
---

## 9. FP/FN 해석

- FP 의미: 실제 완료 주문을 취소 위험 주문으로 잘못 예측하는 경우
- FN 의미: 실제 취소 주문을 완료 주문으로 잘못 예측하여 취소 위험을 놓치는 경우
- 현재 목적에서 더 부담스러울 수 있는 오류: 취소 위험을 사전에 발견하고 대응하는 것이 주요 목적이라면 FN이 상대적으로 중요하다고 판단한다. 다만 이는 사전 대응이 실제 취소를 줄일 수 있다는 가정에 따른 것이다.
- 그 판단에 추가로 필요한 업무 정보: 주문 취소로 발생하는 손실 규모, 취소 방지를 위한 조치 비용, 불필요한 고객 접촉으로 발생하는 비용, 사전 대응의 실제 효과

![Confusion Matrix](images/step06_confusion_matrix.png)

### 업무·분석적 의미
Final Test에서 실제 취소 주문 13건 중 6건을 탐지하였으나 7건은 놓쳤다. 취소 위험을 사전에 관리하는 업무라면 FN은 대응 기회를 놓칠 가능성을 의미한다.
반면 완료 주문 13건을 취소 위험으로 잘못 분류한 FP는 불필요한 고객 안내, 상담 또는 추가 점검 업무를 유발할 수 있다. 따라서 Recall을 높이는 것만으로는 충분하지 않으며 취소 탐지의 기대효과와 오탐지에 따른 업무 비용을 함께 고려해야 한다.
### 한계와 추가 확인 사항
현재 데이터에는 주문 취소에 따른 실제 손실액이나 사전 대응 비용이 포함되어 있지 않아 FP와 FN의 상대적 비용을 정량적으로 비교할 수 없다. 또한 취소 위험을 예측했다는 사실이 해당 주문의 취소 원인을 밝혀낸 것은 아니다.
향후 주문 금액별 취소 손실, 대응 비용 및 사전 개입 효과를 추가로 확인하여 실제 업무 목적에 맞는 Threshold를 설정할 필요가 있다.
---

## 10. Final Test

- 고정한 모델: Logistic Regression
- 고정한 Threshold: 0.45
- Accuracy: 0.6000
- Precision: 0.3158
- Recall: 0.4615
- F1: 0.3750
- Validation 대비 변화: Accuracy 0.7000 → 0.6000, Precision 0.4500 → 0.3158, Recall 0.6923 → 0.4615, F1 0.5455 → 0.3750

### 결과 관찰
Final Test 50건 중 30건을 정확하게 분류하여 Accuracy 60.00%를 기록하였다. 실제 취소 주문 13건 중 6건을 탐지하였으며 FN 7건과 FP 13건이 발생하였다.
Validation 대비 모든 주요 지표가 하락하였으며 특히 Recall은 69.23%에서 46.15%로 23.08%p 감소하였다. 또한 Final Test의 Accuracy는 완료 주문 비율인 74%보다 낮게 나타났다.
### 나의 해석과 판단
Final Test 결과를 보고 모델의 일반화 가능성을 어떻게 판단했는지 작성하세요.
Validation에서 F1이 가장 높았던 Logistic Regression과 Threshold 0.45를 고정하여 평가했으나 Final Test에서는 F1이 0.3750으로 하락하였다. 이는 Validation에서 확인한 취소 탐지 성능이 새로운 평가 데이터에 충분히 유지되지 않았음을 의미한다.
다만 Final Test의 취소 주문은 13건에 불과하여 소수의 예측 결과만 달라져도 지표가 크게 변할 수 있다. 또한 이번 실습은 Stratified Random Split을 사용했으므로 실제 미래 주문에 대한 일반화 성능까지 확인한 것은 아니다.
따라서 현재 결과만으로 모델의 안정적인 운영 성능을 입증하기 어렵다고 판단하였다. 향후에는 더 많은 데이터와 시간순 평가를 통해 성능의 일관성을 추가 검증할 필요가 있다.
### 선택 보호 확인
- [O] Test 결과를 보기 전에 모델을 고정했습니다.
- [O] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [O] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

---

## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰
Final Test 50건에 대한 공개용 예측 결과가 생성되었으며 실제 취소 여부, 예측값, 취소 확률, 선택 모델 및 Threshold의 6개 컬럼으로 구성되었다.
내부 진단용 결과에는 원본 데이터 행을 추적하기 위한 `source_index`가 포함되어 있었으나 공개용 결과에서는 해당 컬럼이 제거되었다. 자동 검증에서도 공개 금지 컬럼의 중복이 없는 것으로 확인되었다.
### 나의 해석과 판단
내부 진단 자료는 잘못 예측한 주문을 원본 데이터와 연결하여 오류 원인을 분석하는 데 활용할 수 있다. 반면 공개 결과에 이러한 추적 정보를 포함하면 다른 데이터와 결합하여 개별 주문이나 고객 정보를 식별할 가능성이 있으므로 공개 목적에 필요한 정보만 제공해야 한다.
따라서 내부 분석의 추적 가능성은 유지하되 공개용 Prediction에서는 원본 식별자를 제거하여 불필요한 정보 노출을 최소화하는 것이 적절하다고 판단하였다.
---

## 12. Validation Evidence

다음 Evidence 중 확인한 항목에 체크합니다.

- [O] `ch10_target_distribution.csv`
- [O] `ch10_feature_audit.csv`
- [O] `ch10_merge_checks.csv`
- [O] `ch10_data_quality_checks.csv`
- [O] `ch10_split_summary.csv`
- [O] `ch10_validation_model_comparison.csv`
- [O] `ch10_validation_threshold_metrics.csv`
- [O] `ch10_test_metrics.csv`
- [O] `ch10_confusion_matrix.csv`
- [O] `ch10_classification_validation.csv`

### 자동 Validation 결과
자동 Validation 7개 항목이 모두 PASS하였다. 이진 Target 정의, 금지 Feature 제외, 병합 정합성, 분할별 클래스 존재 여부, Validation 기반 모델·Threshold 선택 및 공개 예측 결과의 식별자 제거가 확인되었다.
다만 원본 데이터에서 가입일과 주문일이 역전된 45건은 별도의 Data Quality Evidence에 기록하였다. 해당 날짜에서 파생한 `days_since_signup`은 모델 입력에서 제외하였으며 원본 데이터의 정합성 문제가 해결된 것으로 간주하지 않았다.
### 자동 PASS가 의미하지 않는 것
자동 Validation PASS는 사전에 정의된 데이터 및 모델링 절차의 검증 조건을 충족했다는 의미이며 모델의 실제 운영 적합성이나 예측 성능을 보장하지는 않는다.
이번 모델은 Final Test에서 F1 0.3750, Recall 0.4615를 기록하여 Validation 대비 성능이 하락하였다. 또한 Stratified Random Split은 미래 시점의 예측 성능을 직접 검증하지 못하며 고객 특성별 성능 차이나 예측의 공정성도 확인되지 않았다.
따라서 실제 운영에 활용하려면 시간순 독립 평가, 더 많은 검증 데이터, 고객 집단별 성능 및 FP/FN 비용을 추가로 검토해야 한다.
---

## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [O] 현재 사용 보류

### 판단 근거
1. Baseline 대비 성능: Validation에서 Logistic Regression은 F1 0.5000, Recall 0.5385를 기록하여 취소 주문을 전혀 탐지하지 못하는 Dummy Most Frequent(F1 0, Recall 0)보다 개선된 결과를 보였다. 다만 Accuracy는 Dummy의 0.7400보다 낮은 0.7200으로 전체 정확도와 취소 탐지 성능을 구분하여 평가할 필요가 있다.

2. Threshold 및 Final Test 성능: Validation에서 F1을 기준으로 Threshold 0.45를 선택하였으나 Final Test에서는 Precision 0.3158, Recall 0.4615, F1 0.3750으로 성능이 하락하였다. 실제 취소 주문 13건 중 7건을 놓쳤고(FN) 완료 주문 13건을 취소 위험으로 잘못 예측하였다(FP). 현재 성능만으로는 안정적인 취소 위험 탐지를 기대하기 어렵다.

3. 데이터 품질 및 검증 한계: 자동 Validation 7개 항목이 모두 PASS하였으며 독립 스크립트에서도 동일한 결과가 재현되었다. 그러나 원본 데이터에서 발견된 날짜 역전 45건으로 인해 `days_since_signup`을 Feature에서 제외하였고 Stratified Random Split은 실제 미래 주문에 대한 성능을 검증하지 못한다. 따라서 절차적 검증의 통과와 운영 적합성은 구분하여 판단해야 한다.

### 현재 모델의 가장 큰 위험
취소 위험이 있는 주문을 충분히 탐지하지 못하면서도 정상 주문에 불필요한 경고를 발생시킨다는 점이다. 실제 취소 주문의 53.85%를 놓쳤으며 취소 위험으로 분류한 주문 중 실제 취소 비율도 31.58%에 그쳤다. 이러한 오류는 취소 예방 기회의 손실이나 불필요한 고객 대응 업무로 이어질 수 있다.
### 다음 개선 우선순위
1. 데이터 품질 및 Feature 개선: 가입일과 주문일의 정합성 문제를 해결하고 실제 예측 시점에 활용할 수 있는 고객·주문 정보를 추가 검토한다.
2. 모델 일반화 성능 검증: 더 많은 주문 데이터를 확보하여 시간순 검증(Out-of-Time Test)을 수행하고 기간 및 고객 집단별 성능 차이를 확인한다.
3. 업무 목적에 따른 Threshold 검토: FP와 FN의 실제 비용 및 사전 대응 효과를 확인하고 새로운 Validation 데이터에서 업무상 적절한 Threshold를 평가한다.

### 예측과 인과 구분
이번 분석에서 확인한 것은 고객과 주문의 일부 특성을 이용하여 취소 위험을 예측하는 분류 모델의 성능이다. Validation에서 Logistic Regression이 다른 후보보다 높은 F1을 기록했으나 Final Test에서는 충분한 성능이 유지되지 않았다.
이는 입력 Feature를 이용한 예측 결과일 뿐 특정 결제수단이나 고객 특성이 주문 취소를 직접 유발한다는 인과관계를 입증한 것은 아니다. 취소 원인을 규명하려면 추가적인 업무 데이터와 별도의 인과분석이 필요하다.
---

## 최종 체크

- [O] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [O] `refunded`와 기타 상태를 제외했습니다.
- [O] 예측 시점과 Feature Contract를 정의했습니다.
- [O] target, ID, 사후 정보를 feature에서 제외했습니다.
- [O] `line_total = quantity × unit_price`를 검증했습니다.
- [O] 주문 단위 특징과 병합 관계를 검증했습니다.
- [O] 클래스 비율을 확인했습니다.
- [O] Dummy baseline과 비교했습니다.
- [O] Train / Validation / Test 역할을 구분했습니다.
- [O] 모델을 Validation에서 선택했습니다.
- [O] Threshold를 Validation에서 선택했습니다.
- [O] Final Test 전에 선택을 고정했습니다.
- [O] Precision / Recall / F1을 해석했습니다.
- [O] FP/FN의 의미를 설명했습니다.
- [O] Public prediction에서 식별자를 제거했습니다.
- [O] Validation Evidence를 확인했습니다.
- [O] random split의 한계를 기록했습니다.
- [O] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [O] 최종 Notebook URL을 제출합니다.
